# 01 — Patterns and EDA

Explore distinct patterns in surgery counts by specialty and hospital: weekly and yearly seasonality, holiday dips, and trends.

In [ ]:
# Setup: project root and data path (run data generator if data/surgeries.csv is missing)
from pathlib import Path
import sys
root = Path.cwd()
if (root / "notebooks").exists():
    pass
else:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
data_path = root / "data" / "surgeries.csv"
if not data_path.exists():
    from src.data_generator import generate_surgery_counts
    generate_surgery_counts(output_path=str(data_path))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv(data_path, parse_dates=["date"])
df.head()

## Surgery counts over time by specialty

Different specialties show different levels, trends, and volatility.

In [ ]:
# Aggregate by date and specialty (sum across hospitals)
daily_by_spec = df.groupby(["date", "specialty_name"])["surgery_count"].sum().reset_index()
fig, ax = plt.subplots(figsize=(12, 4))
for name in daily_by_spec["specialty_name"].unique():
    sub = daily_by_spec[daily_by_spec["specialty_name"] == name]
    ax.plot(sub["date"], sub["surgery_count"], label=name, alpha=0.8)
ax.set_xlabel("Date")
ax.set_ylabel("Daily surgery count")
ax.set_title("Daily surgery counts by specialty (all hospitals)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Weekly seasonality: mean count by weekday

Weekends typically show lower elective surgery volume; emergency-related specialties may be flatter.

In [ ]:
df["weekday"] = df["date"].dt.dayofweek
df["day_name"] = df["date"].dt.day_name()
weekday_means = df.groupby(["specialty_name", "day_name", "weekday"])["surgery_count"].mean().reset_index()
weekday_means = weekday_means.sort_values("weekday")
days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_means["day_name"] = pd.Categorical(weekday_means["day_name"], categories=days_order, ordered=True)
weekday_means = weekday_means.sort_values(["specialty_name", "day_name"])
g = sns.catplot(data=weekday_means, x="day_name", y="surgery_count", col="specialty_name", col_wrap=3, kind="bar", height=2.5, aspect=1.2)
g.set_xticklabels(rotation=45, ha="right")
g.fig.suptitle("Mean daily surgery count by weekday (per specialty)", y=1.02)
plt.show()

## Yearly seasonality and holiday dips

Aggregate by month to see seasonal pattern; mark known holidays to see dips.

In [ ]:
monthly = df.groupby([df["date"].dt.to_period("M"), "specialty_name"])["surgery_count"].sum().reset_index()
monthly["date"] = monthly["date"].dt.to_timestamp()
fig, ax = plt.subplots(figsize=(12, 4))
for name in monthly["specialty_name"].unique():
    sub = monthly[monthly["specialty_name"] == name]
    ax.plot(sub["date"], sub["surgery_count"], label=name, alpha=0.8)
ax.set_xlabel("Month")
ax.set_ylabel("Total surgery count")
ax.set_title("Monthly surgery counts by specialty")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom into one series and mark holidays (example: General Surgery, H1)
from datetime import datetime
sample = df[(df["specialty_name"] == "General Surgery") & (df["hospital_id"] == "H1")].copy()
sample = sample.set_index("date")["surgery_count"]
holiday_dates = ["2022-01-01", "2022-07-04", "2022-12-25", "2022-11-24", "2023-01-01"]
holidays = pd.to_datetime(holiday_dates)
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(sample.index, sample.values, alpha=0.8, label="Daily count")
for h in holidays:
    if h >= sample.index.min() and h <= sample.index.max():
        ax.axvline(h, color="red", alpha=0.5, linestyle="--")
ax.set_xlabel("Date")
ax.set_ylabel("Surgery count")
ax.set_title("General Surgery @ City General — holiday dates marked")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()